# Oracle to Raw and Cdr Parquet

This notebook:
- connects to the local Oracle container over JDBC using a richer TNS descriptor
- enables `spark.openlineage.jdbc.oracle.tns.enabled=true`
- lets you pick any Oracle schema/table
- creates a small demo table if the requested default table does not exist and the schema is empty
- writes a bronze-style parquet dataset into a `raw` folder with an ingestion timestamp
- reads that parquet back and writes a silver-style parquet dataset into a `cdr` folder

The silver step applies generic cleanup so it works across many table shapes:
- normalizes column names to `snake_case`
- trims string columns
- adds a `cdr_record_hash`
- adds a `cdr_processed_at` timestamp


In [8]:
import os
import re

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T


In [9]:
# Oracle connection settings come from docker-compose env vars by default.
ORACLE_HOST = os.getenv("ORACLE_HOST", "oracle")
ORACLE_PORT = os.getenv("ORACLE_PORT", "1521")
ORACLE_SERVICE = os.getenv("ORACLE_SERVICE", "FREEPDB1")
ORACLE_USER = os.getenv("ORACLE_USER", "SPARK_APP").upper()
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "SparkApp123!")

# Old simple descriptor kept here for quick comparison during converter testing.
# TNS_DESCRIPTOR = (
#     f"(DESCRIPTION="
#     f"(ADDRESS=(PROTOCOL=TCP)(HOST={ORACLE_HOST})(PORT={ORACLE_PORT}))"
#     f"(CONNECT_DATA=(SERVICE_NAME={ORACLE_SERVICE}))"
#     f")"
# )

# Active descriptor keeps the same endpoint but adds nested TNS attributes so
# a TNS-to-EZConnect converter has more structure to parse.
TNS_DESCRIPTOR = (
    f"(DESCRIPTION="
    f"(CONNECT_TIMEOUT=5)"
    f"(TRANSPORT_CONNECT_TIMEOUT=3)"
    f"(RETRY_COUNT=2)"
    f"(ADDRESS_LIST="
    f"(LOAD_BALANCE=OFF)"
    f"(FAILOVER=ON)"
    f"(ADDRESS=(PROTOCOL=TCP)(HOST={ORACLE_HOST})(PORT={ORACLE_PORT}))"
    f"(ADDRESS=(PROTOCOL=TCP)(HOST={ORACLE_HOST})(PORT={ORACLE_PORT}))"
    f")"
    f"(CONNECT_DATA="
    f"(SERVER=DEDICATED)"
    f"(SERVICE_NAME={ORACLE_SERVICE})"
    f")"
    f")"
)
JDBC_URL = f"jdbc:oracle:thin:@{TNS_DESCRIPTOR}"
JDBC_PROPS = {
    "user": ORACLE_USER,
    "password": ORACLE_PASSWORD,
    "driver": "oracle.jdbc.OracleDriver",
}

spark = (
    SparkSession.builder
    .appName("openlineage-oracle-raw-to-cdr")
    .config("spark.openlineage.jdbc.oracle.tns.enabled", "true")
    .getOrCreate()
)

# Set log level to WARN to suppress INFO and DEBUG logs
spark.sparkContext.setLogLevel("INFO")

print("Spark version:", spark.version)
print("Oracle JDBC URL:", JDBC_URL)
print(
    "spark.openlineage.jdbc.oracle.tns.enabled:",
    spark.conf.get("spark.openlineage.jdbc.oracle.tns.enabled"),
)


Spark version: 3.5.3
Oracle JDBC URL: jdbc:oracle:thin:@(DESCRIPTION=(CONNECT_TIMEOUT=5)(TRANSPORT_CONNECT_TIMEOUT=3)(RETRY_COUNT=2)(ADDRESS_LIST=(LOAD_BALANCE=OFF)(FAILOVER=ON)(ADDRESS=(PROTOCOL=TCP)(HOST=oracle)(PORT=1521))(ADDRESS=(PROTOCOL=TCP)(HOST=oracle)(PORT=1521)))(CONNECT_DATA=(SERVER=DEDICATED)(SERVICE_NAME=FREEPDB1)))
spark.openlineage.jdbc.oracle.tns.enabled: true


In [10]:
def read_query(query: str):
    return (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("driver", JDBC_PROPS["driver"])
        .option("user", JDBC_PROPS["user"])
        .option("password", JDBC_PROPS["password"])
        .option("dbtable", f"({query}) q")
        .load()
    )


def read_table(schema_name: str, table_name: str):
    schema_name = schema_name.upper()
    table_name = table_name.upper()
    return (
        spark.read
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("driver", JDBC_PROPS["driver"])
        .option("user", JDBC_PROPS["user"])
        .option("password", JDBC_PROPS["password"])
        .option("dbtable", f"{schema_name}.{table_name}")
        .load()
    )


def write_table(df, schema_name: str, table_name: str, mode: str = "overwrite"):
    schema_name = schema_name.upper()
    table_name = table_name.upper()
    (
        df.write
        .format("jdbc")
        .option("url", JDBC_URL)
        .option("driver", JDBC_PROPS["driver"])
        .option("user", JDBC_PROPS["user"])
        .option("password", JDBC_PROPS["password"])
        .option("dbtable", f"{schema_name}.{table_name}")
        .mode(mode)
        .save()
    )


def to_snake_case(name: str) -> str:
    name = re.sub(r"[^0-9A-Za-z]+", "_", name)
    name = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", name)
    name = re.sub(r"_+", "_", name)
    name = name.strip("_").lower()
    return name or "col"


def normalized_column_names(columns):
    seen = {}
    normalized = []
    for column_name in columns:
        base_name = to_snake_case(column_name)
        seen[base_name] = seen.get(base_name, 0) + 1
        if seen[base_name] == 1:
            normalized.append(base_name)
        else:
            normalized.append(f"{base_name}_{seen[base_name]}")
    return normalized


def seed_demo_table(schema_name: str, table_name: str):
    demo_df = spark.createDataFrame(
        [
            (1001, "+995555000001", "+995555000101", "voice", 62, "2026-03-26 09:00:00", "TBS"),
            (1002, "+995555000002", "+995555000102", "sms", 0, "2026-03-26 09:05:00", "KUT"),
            (1003, "+995555000003", "+995555000103", "data", 245, "2026-03-26 09:10:00", "BAT"),
        ],
        [
            "CDR_ID",
            "CALLING_NO",
            "CALLED_NO",
            "EVENT_TYPE",
            "DURATION_SEC",
            "EVENT_TS",
            "CELL_ID",
        ],
    ).withColumn("EVENT_TS", F.to_timestamp("EVENT_TS"))

    write_table(demo_df, schema_name, table_name, mode="overwrite")


In [11]:
# Change these if you want to target a specific schema/table.
SOURCE_SCHEMA = os.getenv("SOURCE_SCHEMA", ORACLE_USER).upper()
REQUESTED_SOURCE_TABLE = os.getenv("SOURCE_TABLE", "employees_demo2").upper()

available_tables_df = read_query(f"""
    SELECT owner, table_name
    FROM all_tables
    WHERE owner = UPPER('{SOURCE_SCHEMA}')
    ORDER BY table_name
""")

available_tables = [row["TABLE_NAME"] for row in available_tables_df.collect()]
available_tables_df.show(200, truncate=False)

if REQUESTED_SOURCE_TABLE in available_tables:
    SOURCE_TABLE = REQUESTED_SOURCE_TABLE
    print("Using requested source table:", f"{SOURCE_SCHEMA}.{SOURCE_TABLE}")
elif available_tables:
    SOURCE_TABLE = available_tables[0]
    print(
        f"Requested table {SOURCE_SCHEMA}.{REQUESTED_SOURCE_TABLE} was not found. "
        f"Falling back to existing table {SOURCE_SCHEMA}.{SOURCE_TABLE}."
    )
elif SOURCE_SCHEMA == ORACLE_USER:
    SOURCE_TABLE = REQUESTED_SOURCE_TABLE
    print(f"No tables found in {SOURCE_SCHEMA}. Creating demo table {SOURCE_SCHEMA}.{SOURCE_TABLE}.")
    seed_demo_table(SOURCE_SCHEMA, SOURCE_TABLE)
else:
    raise RuntimeError(
        f"No tables found in {SOURCE_SCHEMA}, and automatic demo table creation is only enabled for {ORACLE_USER}."
    )

SOURCE_OBJECT = f"{SOURCE_SCHEMA}.{SOURCE_TABLE}"
print("Source object:", SOURCE_OBJECT)


+---------+---------------+
|OWNER    |TABLE_NAME     |
+---------+---------------+
|SPARK_APP|EMPLOYEES_DEMO |
|SPARK_APP|EMPLOYEES_DEMO2|
|SPARK_APP|LINEAGE_DEMO   |
+---------+---------------+

Using requested source table: SPARK_APP.EMPLOYEES_DEMO2
Source object: SPARK_APP.EMPLOYEES_DEMO2


In [12]:
source_df = read_table(SOURCE_SCHEMA, SOURCE_TABLE)

print("Source row count:", source_df.count())
source_df.printSchema()
source_df.show(20, truncate=False)


Source row count: 4
root
 |-- EMP_ID: decimal(5,0) (nullable = true)
 |-- FIRST_NAME: string (nullable = true)
 |-- LAST_NAME: string (nullable = true)
 |-- DEPARTMENT: string (nullable = true)
 |-- HIRE_DATE: timestamp (nullable = true)

+------+----------+---------+----------+-------------------+
|EMP_ID|FIRST_NAME|LAST_NAME|DEPARTMENT|HIRE_DATE          |
+------+----------+---------+----------+-------------------+
|1001  |Ana       |Kapanadze|IT        |2022-01-15 00:00:00|
|1002  |David     |Smith    |HR        |2021-07-10 00:00:00|
|1003  |Nino      |Beridze  |Finance   |2020-03-05 00:00:00|
|1004  |George    |Brown    |Marketing |2023-11-20 00:00:00|
+------+----------+---------+----------+-------------------+



In [13]:
data_root = "/home/jovyan/work/data"
raw_root = os.path.join(data_root, "raw")
cdr_root = os.path.join(data_root, "cdr")
table_slug = f"{SOURCE_SCHEMA.lower()}__{SOURCE_TABLE.lower()}"

bronze_path = os.path.join(raw_root, table_slug)
silver_path = os.path.join(cdr_root, table_slug)

os.makedirs(raw_root, exist_ok=True)
os.makedirs(cdr_root, exist_ok=True)

bronze_df = (
    source_df
    .withColumn("bronze_source_schema", F.lit(SOURCE_SCHEMA))
    .withColumn("bronze_source_table", F.lit(SOURCE_TABLE))
    .withColumn("bronze_ingested_at", F.current_timestamp())
)

bronze_df.write.mode("overwrite").parquet(bronze_path)

print("Bronze parquet saved to:", bronze_path)
bronze_path


Bronze parquet saved to: /home/jovyan/work/data/raw/spark_app__employees_demo2


'/home/jovyan/work/data/raw/spark_app__employees_demo2'

In [14]:
bronze_parquet_df = spark.read.parquet(bronze_path)

silver_df = bronze_parquet_df.toDF(*normalized_column_names(bronze_parquet_df.columns))

string_columns = [field.name for field in silver_df.schema.fields if isinstance(field.dataType, T.StringType)]
for column_name in string_columns:
    silver_df = silver_df.withColumn(column_name, F.trim(F.col(column_name)))

hash_input_columns = list(silver_df.columns)
silver_df = (
    silver_df
    .withColumn(
        "cdr_record_hash",
        F.sha2(
            F.concat_ws(
                "||",
                *[F.coalesce(F.col(column_name).cast("string"), F.lit("")) for column_name in hash_input_columns]
            ),
            256,
        ),
    )
    .withColumn("cdr_processed_at", F.current_timestamp())
    .dropDuplicates(["cdr_record_hash"])
)

silver_df.write.mode("overwrite").parquet(silver_path)

print("Silver parquet saved to:", silver_path)
print("Silver row count:", silver_df.count())
silver_df.printSchema()
silver_df.show(20, truncate=False)
silver_path


Silver parquet saved to: /home/jovyan/work/data/cdr/spark_app__employees_demo2
Silver row count: 4
root
 |-- emp_id: decimal(5,0) (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- hire_date: timestamp (nullable = true)
 |-- bronze_source_schema: string (nullable = true)
 |-- bronze_source_table: string (nullable = true)
 |-- bronze_ingested_at: timestamp (nullable = true)
 |-- cdr_record_hash: string (nullable = true)
 |-- cdr_processed_at: timestamp (nullable = false)

+------+----------+---------+----------+-------------------+--------------------+-------------------+--------------------------+----------------------------------------------------------------+------------------------+
|emp_id|first_name|last_name|department|hire_date          |bronze_source_schema|bronze_source_table|bronze_ingested_at        |cdr_record_hash                                                 |cdr_processed_a

'/home/jovyan/work/data/cdr/spark_app__employees_demo2'

## Notes

- The Oracle JDBC URL in this notebook uses a richer TNS descriptor, not EZConnect.
- The active TNS string keeps the same local Oracle target but adds `ADDRESS_LIST`, timeout, retry, and `SERVER=DEDICATED` attributes so you can test TNS-to-EZConnect conversion.
- `spark.openlineage.jdbc.oracle.tns.enabled` is set to `true` in the notebook Spark session.
- Change `SOURCE_SCHEMA` and `REQUESTED_SOURCE_TABLE` to target a different Oracle table.
- If the requested table is missing, the notebook falls back to the first existing table in the schema.
- If the schema is empty and matches `SPARK_APP`, the notebook creates a small `LINEAGE_DEMO` table for you.
- `raw/` is used as the bronze landing zone.
- `cdr/` is used as the silver output zone.
